In [1]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# -------------------------------
# 1. Paths to your dataset
# -------------------------------
train_dir = "/workspace/data/raw"   # inside you have "class0" and "class1" folders
val_dir   = "/workspace/data/val"

# -------------------------------
# 2. Base data generators
# -------------------------------
# Basic rescaling
datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),   # change depending on your model
    batch_size=32,
    class_mode="binary"
)

val_gen = datagen.flow_from_directory(
    val_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode="binary"
)

# -------------------------------
# 3. Augmentation generator (for minority class)
# -------------------------------
augment_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

# Example: augment only "class0" (minority class)
minority_dir = os.path.join(train_dir, "class0")

minority_aug_gen = augment_gen.flow_from_directory(
    train_dir,
    classes=["class0"],  # ONLY augment minority class
    target_size=(128, 128),
    batch_size=32,
    class_mode="binary"
)

# -------------------------------
# 4. Combine both generators into a balanced generator
# -------------------------------
def balanced_generator(minority_gen, majority_gen):
    while True:
        x_min, y_min = next(minority_gen)
        x_maj, y_maj = next(majority_gen)
        # Concatenate minority + majority
        x = np.concatenate([x_min, x_maj])
        y = np.concatenate([y_min, y_maj])
        yield x, y

# Majority class generator (no augmentation, just normal flow)
majority_gen = datagen.flow_from_directory(
    train_dir,
    classes=["class1"],   # Majority class
    target_size=(128, 128),
    batch_size=32,
    class_mode="binary"
)

train_balanced_gen = balanced_generator(minority_aug_gen, majority_gen)

# -------------------------------
# 5. Example model
# -------------------------------
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")  # binary classification
])

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

# -------------------------------
# 6. Train with balanced generator
# -------------------------------
steps_per_epoch = min(len(minority_aug_gen), len(majority_gen))  # keep balance

history = model.fit(
    train_balanced_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    epochs=20
)


2025-10-08 11:29:23.585655: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Found 41398 images belonging to 2 classes.


FileNotFoundError: [Errno 2] No such file or directory: '/workspace/data/val'